In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

# The OpenAI SDK pointed at Groq's URL — this is how you reach the Responses API
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-120b"

![HuggingFace researcher flow](hf_researcher_responses_mcp_flow.svg)

Your "skill" (reusable system prompt for report formatting)

In [2]:
RESEARCHER_SKILL = """You are a Hugging Face model research specialist.
When reporting on models, always:
- Lead with a one-line verdict naming the best pick and why
- Present findings as a markdown table (Model | Task | Downloads | Likes | Score)
- End with a short 'Recommendation' paragraph
- Only use data returned by the tools
"""

print(RESEARCHER_SKILL)

You are a Hugging Face model research specialist.
When reporting on models, always:
- Lead with a one-line verdict naming the best pick and why
- Present findings as a markdown table (Model | Task | Downloads | Likes | Score)
- End with a short 'Recommendation' paragraph
- Only use data returned by the tools



In [3]:
TOOLS = [
    {
        "type": "mcp",
        "server_label": "Huggingface",
        "server_url": "https://huggingface.co/mcp",
        "server_description": "Search and access AI models, datasets, and spaces from Hugging Face Hub",
        "require_approval": "never",
    }
]

In [4]:
def research_models(query: str):
    """
    Research Hugging Face models based on a query and return a markdown table of findings.
    """
    response = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": RESEARCHER_SKILL},
            {"role": "user", "content": f"Query: {query}"},
        ],
        tools=TOOLS,
    )
    return response

In [9]:
response = research_models(
    "Find the top 5 trending text-generation models this month on Hugging Face. "
    "Use the hub_repo_search tool and do NOT pass author=null. "
    "If author is not required, leave it blank as an empty string or omit it entirely."
)

# Some SDK versions don't populate .output_text reliably, so fall back to the structured output
report = getattr(response, "output_text", None)
if not report:
    parts = []
    for item in getattr(response, "output", []) or []:
        if getattr(item, "type", None) == "message":
            for block in getattr(item, "content", []):
                text = getattr(block, "text", None)
                if text is None and isinstance(block, dict):
                    text = block.get("text")
                if text:
                    parts.append(text)
    report = "\n".join(parts)

display(Markdown(report))

**Verdict:** **zai-org/GLM-5.3-Flash** is the best pick – it leads the list with the highest downloads (≈190 K), the most likes (1 544) and the top trending score, indicating strong community interest and usage.

| Model | Task | Downloads | Likes | Score |
|-------|------|-----------|-------|-------|
| zai-org/GLM-5.3-Flash | text‑generation | 189.8K | 1544 | 1481 |
| zai-org/GLM-5.3 | text‑generation | 8.8K | 1194 | 1155 |
| OBLITERATUS/Qwen3.8-27B-OBLITERATED | text‑generation | 645.6K | 885 | 333 |
| tencent/Hy4-preview | text‑generation | 1.4K | 258 | 256 |
| unsloth/GLM-5.3-Flash-GGUF | text‑generation | 27.3K | 254 | 252 |

**Recommendation**  
For projects needing cutting‑edge text generation, start with **zai-org/GLM-5.3-Flash** to benefit from its broad adoption, robust community support, and strong performance signals. The other models are also noteworthy—particularly OBLITERATUS/Qwen3.8‑27B‑OBLITERATED for its massive download count and the lightweight GGUF variant from unsloth for efficient deployment.

In [10]:
def show_agent_trace(response):
    items = getattr(response, "output", None)
    if items is None:
        print("This response does not have an .output list. Check that you are using the Responses API.")
        return

    for item in items:
        kind = getattr(item, "type", None)

        if kind == "mcp_list_tools":
            print(f"🔧 DISCOVERED TOOLS")
            tools = getattr(item, "tools", [])
            for tool in tools:
                name = tool.get("name") if isinstance(tool, dict) else getattr(tool, "name", str(tool))
                print(f"    - {name}")

        elif kind == "reasoning":
            for block in getattr(item, "content", []):
                text = getattr(block, "text", None)
                if text is None and isinstance(block, dict):
                    text = block.get("text")
                if text:
                    print(f"\n🧠 REASONING:\n    {text}")

        elif kind == "mcp_call":
            name = getattr(item, "name", None)
            args = getattr(item, "arguments", None)
            print(f"\n📞 CALLED: {name}")
            print(f"    args: {args}")

        elif kind == "message":
            print(f"\n✅ FINAL ANSWER:")
            for block in getattr(item, "content", []):
                text = getattr(block, "text", None)
                if text is None and isinstance(block, dict):
                    text = block.get("text")
                if text:
                    print(text)

show_agent_trace(response)

🔧 DISCOVERED TOOLS
    - hf_whoami
    - hub_repo_search
    - hub_repo_details
    - hf_fs

🧠 REASONING:
    We need to query hub_repo_search for trending text-generation models, top 5 this month. Use sort trendingScore? The tool's sort options: trendingScore, downloads, likes, createdAt, lastModified. To get trending models, we can sort by trendingScore and limit 5, filters include ["text-generation"] perhaps. Also repo_types default ["model","dataset"], but we want only models. So set repo_types ["model"], limit 5, sort trendingScore, filters maybe ["text-generation"]. Query blank.

We must not pass author null; we can omit author. Use hub_repo_search.

Let's call.Now need to produce output per developer instructions:

- Lead with one-line verdict naming best pick and why.
- Present findings as markdown table: Model | Task | Downloads | Likes | Score
- End with short Recommendation paragraph.
- Only use data returned by tools.

We have data for each model: name (repo ID), Task is te